# 02 — Loading Data
**Goal:** Read CSVs correctly the first time — handling the messy formats that Adobe Analytics and paid platforms actually export.

Topics:
- `read_csv` options you actually need
- Fixing dtypes on load
- Adobe Analytics export format quirks
- Paid platform (Google Ads / Meta) export format quirks
- Saving clean data

In [2]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
from pathlib import Path

# Create sample data directory
Path('data/raw/samples').mkdir(parents=True, exist_ok=True)

## 1. Simulate Adobe Analytics export format
Adobe exports have specific quirks: metadata rows at the top, date in `MM/DD/YYYY`, numbers with thousand separators, percentage columns as strings like `"64.3%"`.

In [3]:
# Simulate what Adobe Analytics actually exports
adobe_raw = """Report Suite: My_Fintech_App
Date Range: 01/01/2024 - 03/31/2024
Generated: 04/01/2024 09:00:00

Date,Visits,Unique Visitors,Page Views,Bounce Rate,Average Time Spent (seconds),Orders,Revenue
01/01/2024,"3,200","2,800","9,600",42.5%,185,96,"$4,800.00"
01/02/2024,"2,800","2,400","8,400",45.1%,172,59,"$2,950.00"
01/03/2024,"3,500","3,100","10,500",38.9%,201,158,"$7,900.00"
01/04/2024,"2,600","2,200","7,800",51.3%,154,47,"$2,350.00"
01/05/2024,"4,100","3,600","12,300",35.2%,223,156,"$7,800.00"
"""

with open('data/raw/samples/adobe_export.csv', 'w') as f:
    f.write(adobe_raw)

print('Adobe sample created')

Adobe sample created


In [4]:
# ❌ Naive read — this will fail or produce garbage
df_bad = pd.read_csv('data/raw/samples/adobe_export.csv')
print(df_bad.head(8))
print()
print(df_bad.dtypes)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 5, saw 8


In [5]:
# ✅ Correct read — skip metadata rows, handle formats
df_adobe = pd.read_csv(
    'data/raw/samples/adobe_export.csv',
    skiprows=4,            # skip the 4 metadata lines at the top
    thousands=',',         # "3,200" → 3200
    parse_dates=['Date'],  # parse date column automatically
    dayfirst=False,        # MM/DD/YYYY format (Adobe default)
)

print(df_adobe)
print()
print(df_adobe.dtypes)

        Date  Visits  Unique Visitors  Page Views Bounce Rate  \
0 2024-01-01    3200             2800        9600       42.5%   
1 2024-01-02    2800             2400        8400       45.1%   
2 2024-01-03    3500             3100       10500       38.9%   
3 2024-01-04    2600             2200        7800       51.3%   
4 2024-01-05    4100             3600       12300       35.2%   

   Average Time Spent (seconds)  Orders    Revenue  
0                           185      96  $4,800.00  
1                           172      59  $2,950.00  
2                           201     158  $7,900.00  
3                           154      47  $2,350.00  
4                           223     156  $7,800.00  

Date                            datetime64[us]
Visits                                   int64
Unique Visitors                          int64
Page Views                               int64
Bounce Rate                                str
Average Time Spent (seconds)             int64
Orders  

In [6]:
# Fix remaining messy columns

# Bounce Rate: "42.5%" → 0.425
df_adobe['Bounce Rate'] = (
    df_adobe['Bounce Rate']
    .str.replace('%', '', regex=False)   # remove % sign
    .astype(float) / 100                 # convert to decimal
)

# Revenue: "$4,800.00" → 4800.0
df_adobe['Revenue'] = (
    df_adobe['Revenue']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Rename to snake_case — much easier to work with
df_adobe.columns = [
    'date', 'visits', 'unique_visitors', 'pageviews',
    'bounce_rate', 'avg_time_sec', 'orders', 'revenue'
]

print(df_adobe)
print()
print(df_adobe.dtypes)

        date  visits  unique_visitors  pageviews  bounce_rate  avg_time_sec  \
0 2024-01-01    3200             2800       9600        0.425           185   
1 2024-01-02    2800             2400       8400        0.451           172   
2 2024-01-03    3500             3100      10500        0.389           201   
3 2024-01-04    2600             2200       7800        0.513           154   
4 2024-01-05    4100             3600      12300        0.352           223   

   orders  revenue  
0      96   4800.0  
1      59   2950.0  
2     158   7900.0  
3      47   2350.0  
4     156   7800.0  

date               datetime64[us]
visits                      int64
unique_visitors             int64
pageviews                   int64
bounce_rate               float64
avg_time_sec                int64
orders                      int64
revenue                   float64
dtype: object


## 2. Simulate paid platform export (Google Ads / Meta style)

In [7]:
paid_raw = """Day,Campaign,Impressions,Clicks,Cost,Conversions,Conv. value
2024-01-01,Brand_Search,"45,200",1250,"$620.50",38,"$1,900.00"
2024-01-01,Non_Brand_Search,"120,400",2100,"$1,450.00",21,"$1,050.00"
2024-01-02,Brand_Search,"42,100",1180,"$590.00",35,"$1,750.00"
2024-01-02,Non_Brand_Search,"118,200",2050,"$1,420.00",19,"$950.00"
2024-01-03,Brand_Search,"51,300",1400,"$710.00",52,"$2,600.00"
2024-01-03,Non_Brand_Search,"135,000",2300,"$1,600.00",29,"$1,450.00"
"""

with open('data/raw/samples/paid_export.csv', 'w') as f:
    f.write(paid_raw)

print('Paid sample created')

Paid sample created


In [8]:
df_paid = pd.read_csv(
    'data/raw/samples/paid_export.csv',
    thousands=',',
    parse_dates=['Day'],
)

# Fix currency columns
for col in ['Cost', 'Conv. value']:
    df_paid[col] = (
        df_paid[col]
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )

# Rename to snake_case
df_paid.columns = ['date', 'campaign', 'impressions', 'clicks', 'cost', 'conversions', 'conv_value']

# Add useful derived metrics right on load
df_paid['ctr']      = df_paid['clicks'] / df_paid['impressions'] * 100        # Click-through rate
df_paid['cpc']      = df_paid['cost'] / df_paid['clicks']                     # Cost per click
df_paid['cpa']      = df_paid['cost'] / df_paid['conversions']                # Cost per acquisition
df_paid['roas']     = df_paid['conv_value'] / df_paid['cost']                 # Return on ad spend

print(df_paid.to_string())
print()
print(df_paid.dtypes)

        date          campaign  impressions  clicks    cost  conversions  conv_value       ctr       cpc        cpa      roas
0 2024-01-01      Brand_Search        45200    1250   620.5           38      1900.0  2.765487  0.496400  16.328947  3.062047
1 2024-01-01  Non_Brand_Search       120400    2100  1450.0           21      1050.0  1.744186  0.690476  69.047619  0.724138
2 2024-01-02      Brand_Search        42100    1180   590.0           35      1750.0  2.802850  0.500000  16.857143  2.966102
3 2024-01-02  Non_Brand_Search       118200    2050  1420.0           19       950.0  1.734349  0.692683  74.736842  0.669014
4 2024-01-03      Brand_Search        51300    1400   710.0           52      2600.0  2.729045  0.507143  13.653846  3.661972
5 2024-01-03  Non_Brand_Search       135000    2300  1600.0           29      1450.0  1.703704  0.695652  55.172414  0.906250

date           datetime64[us]
campaign                  str
impressions             int64
clicks                  int

## 3. Useful `read_csv` options cheatsheet

In [9]:
# Full read_csv options you'll actually use:

df = pd.read_csv(
    'data/raw/funnel_data.csv',

    # --- Rows ---
    # skiprows=4,            # skip N rows at the top (Adobe metadata)
    # nrows=1000,            # only read first N rows (for testing)
    # skipfooter=2,          # skip N rows at the bottom

    # --- Columns ---
    # usecols=['date','visits','orders'],  # only load these columns (faster)

    # --- Types ---
    parse_dates=['date'],  # parse these columns as datetime
    # dtype={'channel': 'category'},       # force dtype (saves memory)

    # --- Format ---
    # thousands=',',         # treat comma as thousand separator
    # decimal='.',           # decimal point character
    # encoding='utf-8',      # for special characters (ñ, tildes, etc.)
    # sep=';',               # separator (default is ',')
)

print(df.shape)
print(df.dtypes)

(450, 12)
date                     datetime64[us]
channel                             str
visita_landing                    int64
inicio_solicitud                  int64
datos_personales                  int64
otp                               int64
datos_financieros                 int64
carga_documentos                  int64
evaluacion_crediticia             int64
aprobacion                        int64
firma_digital                     int64
activacion_tarjeta                int64
dtype: object


## 4. Saving clean data

In [10]:
Path('data/clean').mkdir(exist_ok=True)

# Save to CSV — index=False avoids writing the row numbers as a column
df_adobe.to_csv('data/clean/adobe_clean.csv', index=False)
df_paid.to_csv('data/clean/paid_clean.csv', index=False)

# Parquet — much faster to read back, smaller file, preserves dtypes
# Use this when you'll reload the same data many times
df_adobe.to_parquet('data/clean/adobe_clean.parquet', index=False)
df_paid.to_parquet('data/clean/paid_clean.parquet', index=False)

print('Saved to data/clean/')

# Parquet preserves dtypes — no need to re-parse dates on load
df_reload = pd.read_parquet('data/clean/adobe_clean.parquet')
print(df_reload.dtypes)

Saved to data/clean/
date               datetime64[us]
visits                      int64
unique_visitors             int64
pageviews                   int64
bounce_rate               float64
avg_time_sec                int64
orders                      int64
revenue                   float64
dtype: object


## Summary
| Option | What it does |
|---|---|
| `skiprows=N` | Skip metadata rows at the top (Adobe) |
| `thousands=','` | Parse `"3,200"` as `3200` |
| `parse_dates=['col']` | Auto-parse date columns |
| `usecols=[...]` | Only load specific columns (faster) |
| `dtype={'col': type}` | Force a column's dtype on load |
| `encoding='utf-8'` | Handle special characters |
| `.str.replace('$','')` | Clean currency strings |
| `.astype(float)` | Convert string column to number |
| `to_parquet()` | Save with dtypes preserved — fast reload |
| `to_csv(index=False)` | Save without row numbers |

**Next:** `03_selecting_filtering.ipynb` — loc, iloc, boolean indexing, query.